This notebook contains the implementation used for our rotated-MNIST experiments. It trains matched Euclidean and spherical auto-encoder variants—AE, VAE, S-AE, and S-VAE—together with a latent classifier that serves as the frozen oracle. It then learns an oracle-preserving latent generator and visualizes both coordinate-based latent traversals and trajectories obtained by numerically integrating the learned generator. The spherical models use blockwise normalization, tangent-space projection, and exponential-map updates to keep their trajectories on \(S^2\) or \(S^2 * S^2\).
Acknowledgments and code provenance

The oracle-preserving generator, its training objective, and parts of the experimental structure were adapted from the public Oracle-Preserving Latent Flows repository (https://github.com/royforestano/Deep_Learning_Symmetries) accompanying the work of Roman et al. The hyperspherical VAE implementation was informed by the public S-VAE PyTorch repository (https://github.com/nicola-decao/s-vae-pytorch) accompanying the work of Davidson et al. We modified and extended these components to support a shared convolutional architecture, product-sphere latent spaces, geometry-aware generator integration, and the comparisons presented here. The notebook also uses PyTorch, Torchvision, scikit-learn, and the MNIST dataset.

In [ ]:
import numpy as np
import os
import random
import math
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
# from torch.utils.data import DataLoader
# from torch.utils.data import Dataset
from torchvision.datasets import MNIST
# from torchvision import transforms
import torch.nn.functional as F

# from tqdm import tqdm
from sklearn.model_selection import train_test_split
import copy
from itertools import combinations


from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.datasets as Datasets
import torchvision.transforms as transforms
import torch.nn.functional as F
import torchvision.models as models
import torchvision.utils as vutils

from IPython.display import clear_output
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import trange, tqdm
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF



def set_seed(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(15)


batchSize = 64
lr = 1e-4
nepoch = 10
root = "/datasets"
RUN_NAME = 'default'  # don't forget to add corresponding folders for this.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#plt.style.use('dark_background')


In [ ]:
# Load Data:
mnist_dataset = MNIST(root='', train=True,
                      transform=transforms.ToTensor(), download=True)

class MyDataset(Dataset):
    def __init__(self, x, y):
        super(MyDataset, self).__init__()
        assert x.shape[0] == y.shape[0] # assuming shape[0] = dataset size
        self.x = x
        self.y = y

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, index):
        return self.x[index], self.y[index]


def rotate_images(x, start=-75, end=75):
    x = x.to(device)
    angles = torch.randint(low=start, high=end + 1, size=(len(x),), device=device)

    x_rot = torch.empty_like(x)
    for angle in range(start, end + 1):
        mask = (angles == angle)
        if mask.any():
            x_rot[mask] = TF.rotate(x[mask], angle=angle, fill=[0.0])

    return x_rot, angles


def rotate_images_discrete(x, n=4):
    x = x.to(device)
    step = 360 // n
    possible_angles = torch.arange(0, 360, step, device=device)  # e.g. [0, 90, 180, 270]

    idx = torch.randint(low=0, high=n, size=(len(x),), device=device)
    angles = possible_angles[idx]

    x_rot = torch.empty_like(x)
    for angle in possible_angles.tolist():
        mask = (angles == angle)
        if mask.any():
            x_rot[mask] = TF.rotate(x[mask], angle=angle, fill=[0.0])

    return x_rot, angles


max_num = 4
min_num = 3
mask = (mnist_dataset.targets >= min_num) & (mnist_dataset.targets <= max_num)
x = mnist_dataset.data[mask][:, None, :, :].float() / 255
y = mnist_dataset.targets[mask][:, None].float()
y = (y == max_num).float()  # 1 -> 0, 2 -> 1

# x, angles = rotate_images(x, start=-75, end=75)
x, angles = rotate_images_discrete(x, n=8)

# Normalize to [-1, 1]
x = (x - 0.5) / 0.5

# Normalize to [0, 1]
# x = x / 1

x_train, x_test, y_train, y_test = train_test_split(
    x.cpu(), y, test_size=.25, shuffle=True, random_state=0
)

train_dataset = MyDataset(x_train, y_train)
test_dataset  = MyDataset(x_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

dataiter = iter(test_loader)
test_images = next(dataiter)[0]


In [ ]:
# @title Auto-Encoders

def sample(model, epoch=-1, run_name='default'):
    plt.figure(figsize=[12,5])

    for i, (real, fake, real_test, test) in enumerate(zip(
                                            x_train[:10],
                                            model(x_train[:10].to(device)),
                                            x_test[:10],
                                            model(x_test[:10].to(device)))):

        plt.subplot(4,10,i+1)
        plt.imshow(real.detach().numpy().squeeze(), cmap='inferno')
        plt.axis('off')

        plt.subplot(4,10,i+11)
        plt.imshow(fake.detach().cpu().numpy().squeeze(), cmap='inferno')
        plt.axis('off')

        plt.subplot(4,10,i+21)
        plt.imshow(real_test.detach().cpu().numpy().squeeze(), cmap='inferno')
        plt.axis('off')

        plt.subplot(4,10,i+31)
        plt.imshow(test.detach().cpu().numpy().squeeze(), cmap='inferno')
        plt.axis('off')

    # plt.savefig(f'./training_images/{run_name}/epoch_{epoch}.png',
    #             bbox_inches='tight')
    plt.show()



latent_size = 6  # use 3 for S^2 or 6 for S^2 x S^2
SPHERE_BLOCK_DIM = 3
assert latent_size in (3, 6), "Use latent_size=3 (S^2) or latent_size=6 (S^2 x S^2)."


def sphere_blocks(z):
    """View the latent as consecutive three-coordinate S^2 factors."""
    assert z.shape[-1] % SPHERE_BLOCK_DIM == 0
    return z.reshape(*z.shape[:-1], z.shape[-1] // SPHERE_BLOCK_DIM, SPHERE_BLOCK_DIM)


def normalize_sphere_blocks(z):
    """Normalize each S^2 factor independently."""
    return F.normalize(sphere_blocks(z), p=2, dim=-1, eps=1e-6).flatten(-2)
# Deterministic autoencoder reconstruction loss
ae_loss_fn = nn.MSELoss()
KL_WEIGHT = 0.01
AE_LR = 1e-4
AE_EPOCHS = 200



# Model:

class SharedEncoder():
  def __init__(self):
    self.shared_enc = nn.Sequential(
              nn.Conv2d(1, 128, 3, stride=2, padding=1, bias=True),   # 28 -> 14
              nn.ReLU(),
              nn.Conv2d(128, 64, 3, stride=2, padding=1, bias=True),  # 14 -> 7
              nn.ReLU(),
              nn.Conv2d(64, 32, 3, stride=2, padding=1, bias=True),   # 7 -> 4
              nn.ReLU(),
              nn.Flatten(),
              nn.Linear(32 * 4 * 4, 256),
              nn.ReLU(),
              )

class SharedDecoder():
  def __init__(self):
    self.shared_dec = nn.Sequential(
            nn.ReLU(),
            nn.Linear(256, 32 * 4 * 4),
            nn.ReLU(),
            nn.Unflatten(1, (32, 4, 4)),
            nn.ConvTranspose2d(32, 64, 3, stride=2, padding=1, output_padding=0),   # 4 -> 7
            nn.ReLU(),
            nn.ConvTranspose2d(64, 128, 3, stride=2, padding=1, output_padding=1),  # 7 -> 14
            nn.ReLU(),
            nn.ConvTranspose2d(128, 1, 3, stride=2, padding=1, output_padding=1),   # 14 -> 28
            nn.Tanh()
            )

# AE
class ConvAutoencoder(nn.Module):
    def __init__(self, n_latent=3):
        super(ConvAutoencoder, self).__init__()

        self.layer_enc = nn.Linear(256, n_latent)
        self.layer_dec = nn.Linear(n_latent, 256, bias=True)

        self.enc = SharedEncoder().shared_enc
        self.dec = SharedDecoder().shared_dec

    def encode(self, x):
        return self.layer_enc(self.enc(x))

    def decode(self, z):
        return self.dec(self.layer_dec(z))

    def forward(self, x):
        return self.decode(self.encode(x))

# SAE
class ConvSphericalAutoencoder(nn.Module):
    def __init__(self, n_latent=3, radius=1.0):
        super(ConvSphericalAutoencoder, self).__init__()

        self.layer_enc = nn.Linear(256, n_latent)
        self.layer_dec = nn.Linear(n_latent, 256, bias=True)

        self.radius = radius
        self.enc = SharedEncoder().shared_enc
        self.dec = SharedDecoder().shared_dec

    def encode(self, x):
        z_raw = self.layer_enc(self.enc(x))
        z = normalize_sphere_blocks(z_raw)
        z = self.radius * z
        return z

    def decode(self, z):
        return self.dec(self.layer_dec(z))

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)


# VAE
class Encoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()

        self.conv = SharedEncoder().shared_enc
        self.mu = nn.Linear(256, latent_dim)
        self.logvar = nn.Linear(256, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.flatten(1)

        mu = self.mu(x)
        logvar = self.logvar(x)

        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std

        return z, mu, logvar

class Decoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()

        self.fc = nn.Linear(latent_dim, 256)
        self.dec = SharedDecoder().shared_dec

    def forward(self, z):
        x = self.fc(z)
        # x = x.view(-1, 128, 4, 4)
        return self.dec(x)


class VAE(nn.Module):
    def __init__(self, n_latent=32):
        super(VAE, self).__init__()
        self.encode = Encoder(latent_dim=n_latent)
        self.decode = Decoder(latent_dim=n_latent)

    def forward(self, x):
        encoding, mu, logvar = self.encode(x)

        if self.training:
            x = self.decode(encoding)
        else:
            x = self.decode(mu)

        return x, encoding, mu, logvar


# S-VAE
def sample_vmf_s2(mu, kappa):
    """Differentiable inverse-CDF sample from vMF(mu, kappa) on S^2."""
    # Avoid exact CDF endpoints, where the sqrt derivative is singular.
    sample_eps = 1e-6
    u = torch.rand_like(kappa).clamp(sample_eps, 1.0 - sample_eps)
    w = 1.0 + torch.log(
        u + (1.0 - u) * torch.exp(-2.0 * kappa)
    ) / kappa
    w = w.clamp(-1.0 + sample_eps, 1.0 - sample_eps)
    phi = 2.0 * math.pi * torch.rand_like(kappa)
    radial = torch.sqrt((1.0 - w) * (1.0 + w))
    north_sample = torch.cat(
        (radial * torch.cos(phi), radial * torch.sin(phi), w), dim=-1
    )

    north = torch.zeros_like(mu)
    north[:, -1] = 1.0
    axis = north - mu
    axis_norm = axis.norm(dim=-1, keepdim=True)
    unit_axis = axis / axis_norm.clamp_min(sample_eps)
    reflected = north_sample - 2.0 * (
        north_sample * unit_axis
    ).sum(-1, keepdim=True) * unit_axis
    sample = torch.where(axis_norm > sample_eps, reflected, north_sample)
    return F.normalize(sample, p=2, dim=-1, eps=sample_eps)


def vmf_uniform_kl_s2(kappa):
    """KL(vMF(mu, kappa) || Uniform(S^2)); independent of mu."""
    log_sinh = kappa + torch.log1p(-torch.exp(-2.0 * kappa)) - math.log(2.0)
    return torch.log(kappa) - log_sinh + kappa / torch.tanh(kappa) - 1.0


class SVAE(nn.Module):
    def __init__(self, z_dim, activation=F.relu, dims=784, img_size=28):
        super().__init__()

        assert z_dim in (3, 6), "Use z_dim=3 (S^2) or z_dim=6 (S^2 x S^2)."
        self.z_dim = z_dim
        self.n_spheres = z_dim // SPHERE_BLOCK_DIM
        self.encoder = SharedEncoder().shared_enc
        self.fc_mean = nn.Linear(256, z_dim)
        self.fc_var = nn.Linear(256, self.n_spheres)
        self.fc_decode = nn.Linear(z_dim, 256)
        self.decoder = SharedDecoder().shared_dec

    def encode(self, x):
        x = x.reshape(-1, 1, 28, 28)
        features = self.encoder(x)
        z_mean = normalize_sphere_blocks(self.fc_mean(features))
        z_var = F.softplus(self.fc_var(features)) + 1.0
        return z_mean, z_var

    def decode(self, z):
        return self.decoder(self.fc_decode(z))

    def forward(self, x):
        z_mean, z_var = self.encode(x)
        if self.training:
            mean_blocks = sphere_blocks(z_mean).reshape(-1, SPHERE_BLOCK_DIM)
            sampled_blocks = sample_vmf_s2(
                mean_blocks, z_var.reshape(-1, 1)
            )
            z = sampled_blocks.reshape(-1, self.n_spheres, SPHERE_BLOCK_DIM).flatten(-2)
        else:
            z = z_mean
        reconstruction = self.decode(z)
        return reconstruction, (z_mean, z_var), z


In [ ]:
def train_func(auto, epochs, train_loader, device, loss_fn, optimizer):
  pbar = trange(epochs, leave=False, desc="Epoch")
  for epoch in pbar:
      auto.train()
      total_sum = recon_sum = kl_sum = 0.0

      for images, _ in tqdm(train_loader, leave=False, desc="Training"):
          images = images.to(device)
          output = auto(images)

          if isinstance(auto, (VAE, SVAE)):
              loss, recon_loss, kl_loss = loss_fn(
                  output, images, return_components=True
              )
          else:
              loss = recon_loss = loss_fn(output, images)
              kl_loss = None

          if not torch.isfinite(loss):
              raise FloatingPointError(
                  f"Non-finite loss while training {auto.__class__.__name__}."
              )

          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

          total_sum += loss.item()
          recon_sum += recon_loss.item()
          if kl_loss is not None:
              kl_sum += kl_loss.item()

      n_batches = len(train_loader)
      message = (
          f"Epoch [{epoch + 1}/{epochs}]  "
          f"total={total_sum / n_batches:.6f}  "
          f"recon={recon_sum / n_batches:.6f}"
      )
      if isinstance(auto, (VAE, SVAE)):
          avg_kl = kl_sum / n_batches
          message += f"  KL={avg_kl:.6f}  weighted_KL={KL_WEIGHT * avg_kl:.6f}"
      print(message)


In [ ]:
# @title Train AE

auto_ae = ConvAutoencoder(latent_size).to(device)
optimizer_ae = optim.Adam(
    auto_ae.parameters(),
    lr=AE_LR
)
train_func(auto_ae, AE_EPOCHS, train_loader, device, ae_loss_fn, optimizer_ae)


In [ ]:
# @title Train SAE

auto_sae = ConvSphericalAutoencoder(latent_size).to(device)
optimizer_sae = optim.Adam(
    auto_sae.parameters(),
    lr=AE_LR
)
train_func(auto_sae, AE_EPOCHS, train_loader, device, ae_loss_fn, optimizer_sae)


In [ ]:
# @title Train VAE

# KL Divergence penalty (loss)
def vae_loss(recon, x, return_components=False):
    recon_data, z_latent, mu, logvar = recon

    recon_loss = F.mse_loss(recon_data, x)

    # Here is our KL divergance loss implemented in code
    # We will use the mean across the dimensions instead of the sum (which is common and would require different scaling)
    kl_loss = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).mean()

    # We'll tune the "strength" of KL divergance loss to get a good result
    loss = recon_loss + KL_WEIGHT * kl_loss
    if return_components:
        return loss, recon_loss, kl_loss
    return loss


vae_net = VAE(n_latent=latent_size).to(device)
optimizer_vae = optim.Adam(
    vae_net.parameters(),
    lr=AE_LR
)

train_func(vae_net, AE_EPOCHS, train_loader, device, vae_loss, optimizer_vae)

In [ ]:
# @title Train S-VAE

def svae_loss(parameters, x_mb, return_components=False):
  reconstruction, (z_mean, kappa), _ = parameters
  loss_recon = F.mse_loss(reconstruction, x_mb)
  loss_kl = vmf_uniform_kl_s2(kappa).sum(dim=-1).mean()
  loss = loss_recon + KL_WEIGHT * loss_kl
  if return_components:
    return loss, loss_recon, loss_kl
  return loss


svae_net = SVAE(z_dim=latent_size).to(device)
optimizer_svae = torch.optim.Adam(
    svae_net.parameters(),
    lr=AE_LR
)

train_func(svae_net, AE_EPOCHS, train_loader, device, svae_loss, optimizer_svae)


In [ ]:
def test_func(auto, x_test, device, names=None):
  n = 8
  models = auto if isinstance(auto, (list, tuple)) else [auto]

  with torch.no_grad():
      images = x_test[:n].to(device)
      recons = []
      for model in models:
          model.eval()
          recon = model(images)
          if isinstance(recon, tuple):  # if VAE
              recon = recon[0]
          recons.append(recon.cpu())

  images = images.cpu()

  n_rows = 1 + len(models)
  fig, axes = plt.subplots(n_rows, n, figsize=(n * 1.5, 1.5 * n_rows))

  for i in range(n):
      axes[0, i].imshow(images[i].squeeze(), cmap='gray')
      axes[0, i].axis('off')

  for row, recon in enumerate(recons, start=1):
      for i in range(n):
          axes[row, i].imshow(recon[i].squeeze(), cmap='gray')
          axes[row, i].axis('off')

  axes[0, 0].set_ylabel('Original', fontsize=10)
  for row in range(1, n_rows):
      label = names[row - 1] if names else f'Reconstructed {row}'
      axes[row, 0].set_ylabel(label, fontsize=10)

  plt.tight_layout()
  plt.show()


In [ ]:
print("SVAE:")
test_func([svae_net], x_test, device)

In [ ]:
print("AE:")
test_func([auto_ae], x_test, device)

In [ ]:
# @title Sample from random latent space

def sample_func(auto, latent_size, device, names=None):
  sample_n = 8
  models = auto if isinstance(auto, (list, tuple)) else [auto]

  samples_list = []
  with torch.no_grad():
      for model in models:
          model.eval()

          if isinstance(model, (SVAE, ConvSphericalAutoencoder)):
              sample_z = normalize_sphere_blocks(
                  torch.randn(sample_n, latent_size, device=device)
              )
          else:
              sample_z = torch.randn(sample_n, latent_size).to(device)

          samples = model.decode(sample_z).cpu()
          samples_list.append(samples)

  n_rows = len(models)
  fig, axes = plt.subplots(n_rows, sample_n, figsize=(sample_n * 1.5, 1.5 * n_rows))
  if n_rows == 1:
      axes = axes[None, :]  # keep indexing 2D even for a single model

  for row, samples in enumerate(samples_list):
      for i in range(sample_n):
          axes[row, i].imshow(samples[i].squeeze(), cmap='gray')
          axes[row, i].axis('off')

      label = names[row] if names else (f'Model {row + 1}' if n_rows > 1 else None)
      if label:
          axes[row, 0].set_ylabel(label, fontsize=10)

  plt.tight_layout()
  plt.show()


In [ ]:
# AE
sample_func(auto_ae, latent_size, device)

In [ ]:
# SAE
sample_func(auto_sae, latent_size, device)

In [ ]:
# VAE
sample_func(vae_net, latent_size, device)

In [ ]:
# S-VAE
sample_func(svae_net, latent_size, device)

In [ ]:
def get_z(auto, x):
    auto.eval()

    with torch.no_grad():
        x = x.to(device)

        if isinstance(auto, VAE):
            z, mu, logvar = auto.encode(x)
            z = mu  # deterministic representation during evaluation

        elif isinstance(auto, SVAE):
            z_mean, z_var = auto.encode(x)
            z = z_mean  # point on the sphere

        else:
            z = auto.encode(x)

    return z.detach().cpu()


In [ ]:
z = get_z(auto_ae, x)

In [ ]:
z = get_z(auto_sae, x)

In [ ]:
z = get_z(vae_net, x)

In [ ]:
z = get_z(svae_net, x)

In [ ]:
# Get a test image
dataiter = iter(test_loader)
test_images = next(dataiter)[0]

# Classifier

In [ ]:
# Model:

class Classifier(nn.Module):

    def __init__(self, n_latent=latent_size):
        super(Classifier, self).__init__()

        # Encoder
        self.layers = nn.Sequential(
            nn.Linear(n_latent, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 32), nn.ReLU(),
            # nn.Linear(32, 10),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        # F.softmax(self.layers(x), 1)

        return torch.sigmoid(self.layers(x))

In [ ]:

# Hyperparameters
num_epochs = 60
batch_size = 256
learning_rate = 1e-4


z_train, z_test, y_train, y_test = train_test_split(z, y, test_size=.25, shuffle=True)

train_dataset_z = MyDataset(z_train, y_train)
test_dataset_z  = MyDataset(z_test,  y_test)

train_dataloader_z = DataLoader(train_dataset_z, batch_size=batch_size, shuffle=True)
test_dataloader_z  = DataLoader(test_dataset_z,  batch_size=batch_size, shuffle=True)


In [ ]:

def train_classifier(model, learning_rate, num_epochs, train_dataloader, test_dataloader, device):
  #criterion = nn.CrossEntropyLoss()
  criterion = nn.MSELoss()
  optimizer = optim.Adam(model.parameters(), lr=learning_rate)

  loss_history = np.zeros((num_epochs, 2))
  acc_history = np.zeros((num_epochs, 2))

  # Train the model
  pbar = tqdm(range(num_epochs), desc="Epoch")
  for epoch in pbar:

      train_loss = 0
      train_correct = 0
      for z_batch, y_batch in train_dataloader:

          # print(z_batch.shape, y_batch.shape)
          pred = model(z_batch.to(device))
          loss = criterion(pred, y_batch.to(device))
          train_loss += loss.item()*z_batch.shape[0]
          train_correct += (pred.round() == y_batch.to(device)).sum().item()

          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

      test_loss = 0
      test_correct = 0
      for z_batch, y_batch in test_dataloader:

          pred = model(z_batch.to(device))
          loss = criterion(pred, y_batch.to(device))
          test_loss += loss.item()*z_batch.shape[0]
          test_correct += (pred.round() == y_batch.to(device)).sum().item()

      loss_history[epoch, 0] = train_loss / train_dataloader.dataset.x.shape[0]
      loss_history[epoch, 1] = test_loss / test_dataloader.dataset.x.shape[0]
      acc_history[epoch, 0] = train_correct / train_dataloader.dataset.x.shape[0]
      acc_history[epoch, 1] = test_correct / test_dataloader.dataset.x.shape[0]

      # if epoch % 50 == 0:
      #     sample(model, epoch, RUN_NAME)

      # torch.save(model.state_dict(),
      #            f'./trained_models/{RUN_NAME}/conv_model_weights_test.pth')

      pbar.set_postfix({
          'Train Loss': f'{loss_history[epoch,0]:.4f}',
          'Val Loss': f'{loss_history[epoch,1]:.4f}',
          'Train Acc': f'{acc_history[epoch,0]:.4f}',
          'Val Acc': f'{acc_history[epoch,1]:.4f}'
      })




# model = Classifier()
model = Classifier().to(device)
train_classifier(model, learning_rate, num_epochs, train_dataloader_z, test_dataloader_z, device)


In [ ]:
# Valid only for 2d
plt.scatter(*z_test.cpu().detach().numpy().T, c=y_test.cpu().detach().numpy(), s=2, cmap='RdBu')

In [ ]:
model = model.to(device)

In [ ]:
# @title Ordinary 2d

def inv_sigmoid(x):
    return -np.log(np.maximum((1 / np.maximum(x, 1e-30)) - 1, .0001))

z_all = torch.cat([z_train, z_test]).cpu().detach().numpy()

margin = 0.5
x_min, x_max = z_all[:,0].min() - margin, z_all[:,0].max() + margin
y_min, y_max = z_all[:,1].min() - margin, z_all[:,1].max() + margin

X, Y = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

grid_points = torch.tensor(np.stack([X.flatten(), Y.flatten()]).T).float()
grid_preds = model.layers(grid_points.to(device)).cpu().flatten().detach().numpy()


In [ ]:
vmax = np.abs(grid_preds).max()

plt.contourf(X, Y, grid_preds.reshape(X.shape), 100, cmap='RdBu', vmin=-vmax, vmax=vmax)
plt.scatter(*z_test.cpu().detach().numpy().T, c=y_test.cpu().detach().numpy(), s=2, cmap='RdBu')
plt.colorbar()

In [ ]:
# @title More than 2d

from sklearn.decomposition import PCA

def inv_sigmoid(x):
    return -np.log(np.maximum((1 / np.maximum(x, 1e-30)) - 1, .0001))

z_all = torch.cat([z_train, z_test]).cpu().detach().numpy()

pca = PCA(n_components=2)
z_all_2d = pca.fit_transform(z_all)

margin = 0.01
x_min, x_max = z_all_2d[:,0].min() - margin, z_all_2d[:,0].max() + margin
y_min, y_max = z_all_2d[:,1].min() - margin, z_all_2d[:,1].max() + margin

X, Y = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))


grid_points_full = pca.inverse_transform(np.stack([X.flatten(), Y.flatten()]).T)
grid_points = torch.tensor(grid_points_full).float()
grid_preds = model.layers(grid_points.to(device)).cpu().flatten().detach().numpy()


In [ ]:
vmax = np.abs(grid_preds).max()
plt.contourf(X, Y, grid_preds.reshape(X.shape), 100, cmap='RdBu', vmin=-vmax, vmax=vmax)

z_test_2d = pca.transform(z_test.cpu().detach().numpy())
plt.scatter(*z_test_2d.T, c=y_test.cpu().detach().numpy(), s=2, cmap='RdBu')
plt.colorbar()


# G Network

In [ ]:
def normalized_dot(a,b):
    '''Given a_ij, b_ik returns a_ij b_ij / (|a_i|*|b_i|) (summed only over j!)
    This is equivalent to einsum('ik,jk', a, b).diag(), but avoids needlessly
    computing the off diagonal elements. '''

    return torch.sum(a * b, dim=1) / (torch.norm(a, dim=1) * torch.norm(b, dim=1) + 1e-16)

def make_orthonormal(vecs):

    new_vecs = []

    def normalize(v):
        return v / torch.norm(v)

    for i, v in enumerate(vecs):

        #in_space_comp = torch.zeros((i, v.shape[0]))
        projections = [torch.zeros_like(v)]

        for j, w in enumerate(new_vecs[:i]):
            projections.append(v @ w * w)

        new_vecs.append(normalize(v - torch.stack(projections).sum(dim=0)))

    return torch.stack(new_vecs)


def get_nullspace_component(vec, basis):

    in_subspace_component = []

    for i, b in enumerate(basis):
        in_subspace_component.append(b * torch.dot(vec, b))

    return torch.norm(vec - torch.sum(torch.stack(in_subspace_component), dim=0))


def closure_loss(params):

    generators = [p for p in params]

    vecs = torch.flatten(torch.stack(generators), start_dim=1)
    basis = make_orthonormal(vecs)

    temp = [torch.tensor(0)]

    for i,j in combinations(range(len(generators)), 2):

        brac = bracket(generators[i], generators[j]).flatten()
        temp.append(get_nullspace_component(brac, basis)**2)

    return torch.sum(torch.stack(temp))


def ensemble_loss(inp,
                  model,
                  h_inv=1,
                  h_norm=1,
                  h_orth=0,
                  h_clos=0,
                  eps=1e-3,
                  return_components=False):

    transformed = model(inp, eps)
    loss_inv = inp.new_zeros(())
    loss_norm = inp.new_zeros(())
    loss_orth = inp.new_zeros(())
    loss_clos = inp.new_zeros(())  # closure is disabled in the current model

    for x_tran in transformed:
        displacement = torch.norm(x_tran - inp, dim=1)
        loss_inv += torch.mean((oracle(inp) - oracle(x_tran)) ** 2) / eps ** 2
        loss_norm += (displacement.mean() / eps - 1) ** 2
        loss_norm += displacement.std() / eps

    for i, j in combinations(range(len(transformed)), 2):
        loss_orth += torch.mean(
            normalized_dot(transformed[i] - inp, transformed[j] - inp) ** 2
        )

    components = (
        h_inv * loss_inv,
        h_norm * loss_norm,
        h_orth * loss_orth,
        h_clos * loss_clos,
    )
    total = sum(components)
    return (total, components) if return_components else total


In [ ]:
# @title Gs

class GeneratorModel(nn.Module):
    """Shared G network; spherical=True changes only the geometric step."""
    def __init__(self, n_dim=3, n_latent=16, n_generators=1, spherical=False):
        super().__init__()
        if spherical:
            assert n_dim in (3, 6), "Use 3 coordinates for S^2 or 6 for S^2 x S^2."

        self.n_generators = n_generators
        self.spherical = spherical
        self.W_list = nn.ModuleList([
            nn.Sequential(
                nn.Linear(n_dim, n_latent),
                nn.Tanh(),
                nn.Linear(n_latent, n_dim)
            )
            for _ in range(n_generators)
        ])

    def velocity(self, z, net):
        velocity = net(z)
        if self.spherical:
            z_blocks = sphere_blocks(normalize_sphere_blocks(z))
            velocity_blocks = sphere_blocks(velocity)
            velocity_blocks = velocity_blocks - (
                velocity_blocks * z_blocks
            ).sum(dim=-1, keepdim=True) * z_blocks
            velocity = velocity_blocks.flatten(-2)
        return F.normalize(velocity, p=2, dim=-1)
        # return velocity

    def forward(self, z, epsilon=1e-3):
        outputs = []
        for net in self.W_list:
            velocity = self.velocity(z, net)

            if not self.spherical:
                z_new = z + epsilon * velocity
            else:
                # Product exponential-map step on S^2 or S^2 x S^2.
                z_blocks = sphere_blocks(normalize_sphere_blocks(z))
                velocity_blocks = sphere_blocks(velocity)
                block_speed = velocity_blocks.norm(dim=-1, keepdim=True)
                unit_velocity = velocity_blocks / block_speed.clamp_min(1e-8)

                angle = torch.as_tensor(epsilon, dtype=z.dtype, device=z.device)
                while angle.ndim < block_speed.ndim:
                    angle = angle.unsqueeze(-1)
                angle = angle * block_speed

                new_blocks = (
                    torch.cos(angle) * z_blocks
                    + torch.sin(angle) * unit_velocity
                )
                new_blocks = torch.where(
                    block_speed > 1e-8, new_blocks, z_blocks
                )
                z_new = normalize_sphere_blocks(new_blocks.flatten(-2))

            outputs.append(z_new)
        return outputs


class SphericalGeneratorModel(GeneratorModel):
    def __init__(self, n_dim=3, n_latent=16, n_generators=1):
        super().__init__(
            n_dim=n_dim,
            n_latent=n_latent,
            n_generators=n_generators,
            spherical=True,
        )


In [ ]:
model = model.to('cpu')

def oracle(x):
    return model.layers(x)

In [ ]:
# @title Training G on all latent variables

dataset = z_train.to('cpu')

y = oracle(dataset)

np.random.seed(0)
torch.manual_seed(0)
# generator = GeneratorModel(n_dim=latent_size, n_latent=32).to('cpu')
generator = SphericalGeneratorModel(n_dim=latent_size, n_latent=32).to('cpu')

# At this point you can use `model(dataset)` to get the transformed datasets.
# Though at this point the generators are just these random matricies:

# visualize_generators(model.parameters(), n_generators=n_generators)

# Hyperparameters
lr = .002
n_epochs = 800
eps = 1e-4

h_inv  = 1
h_norm = 1
h_orth = 1
h_clos = 1

# Define the optimizer
optim = torch.optim.Adam(generator.parameters(), lr=lr)

# Keep track of some things
best_loss_so_far = np.inf
loss_history = []
loss_components = []

for epoch in range(1, n_epochs+1):

    # Gradient descent part:
    optim.zero_grad()

    loss, components = ensemble_loss(dataset,
                          generator,
                          h_inv=h_inv,
                          h_norm=h_norm,
                          h_orth=h_orth,
                          h_clos=h_clos,
                          eps=eps,
                          return_components=True)
    loss.backward()
    optim.step()

    # Show and track progress:
    if epoch % 1 == 0:

        values = [float(component.detach()) for component in components]
        print(
            f"epoch {epoch}: total={float(loss.detach()):.6f}  "
            f"inv={values[0]:.6f}  norm={values[1]:.6f}  "
            f"orth={values[2]:.6f}  closure={values[3]:.6f}",
            end="\r",
        )
        loss_history.append((epoch, float(loss.detach())))
        loss_components.append((epoch, *values))

    # Keep track of best model so far:
    if loss < best_loss_so_far:
        best_loss_so_far = loss
        best_model       = copy.deepcopy(generator)
        best_model_epoch = epoch

        # # break training if loss gets really small
        # if loss < stopping_thresh:
        #     print(f'epoch {epoch}: Loss: {float(loss)}', ' '*20)
        #     print('Reached loss near machine zero')
        #     loss_history.append((epoch, float(loss)))
        #     break

else: print('\n', f'Best Loss: {best_loss_so_far} \n')

# ### Visualize the loss during the training process:

# I have broken the loss into the three parts:
#  - Invarience: (controlled by `h_inv`)
#  - Normalization: (controlled by `h_norm`)
#  - Closure: (controlled by `h_orth`)

loss_history = np.array(loss_history)
loss_components = np.array(loss_components)

plt.figure(figsize=[7,5], dpi=100)

plt.plot(loss_components[:,0], loss_components[:,1], label='Invariance')
plt.plot(loss_components[:,0], loss_components[:,2], label='Normalization')
plt.plot(loss_components[:,0], loss_components[:,3], label='Orthogonality')
plt.plot(loss_components[:,0], loss_components[:,4], label='Closure')
plt.plot(loss_history[:,0], loss_history[:,1], ls='--', label='Total')
plt.legend()

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.yscale('log')
plt.title('Components of Loss')

plt.show()

In [ ]:
# Generate an iterative walk using either Euclidean or spherical G.

def encode_for_walk(auto, images):
    encoded = auto.encode(images)
    if isinstance(encoded, tuple):
        # Gaussian VAE: (sample, mean, logvar); S-VAE: (mean, kappa).
        return encoded[1] if len(encoded) == 3 else encoded[0]
    return encoded


@torch.no_grad()
def generator_walk(
    generator,
    auto,
    images,
    n_steps=9000,
    epsilon=1e-3,
    save_every=500,
    device=device,
):
    generator = generator.to(device).eval()
    auto = auto.to(device).eval()

    while images.dim() < 4:
        images = images.unsqueeze(0)
    z = encode_for_walk(auto, images.to(device))

    history = [z.detach().cpu()]
    for step in range(1, n_steps + 1):
        z = generator(z, epsilon=epsilon)[0]
        if step % save_every == 0:
            history.append(z.detach().cpu())

    return torch.stack(history)  # (saved steps, batch, latent dim)


In [ ]:
g_paths = generator_walk(generator, auto_sae, test_images[:3])

In [ ]:
# Visualize a G-network walk for any of the four autoencoders.

@torch.no_grad()
def visualize_generator_walk(paths, auto, device=device):
    auto = auto.to(device).eval()
    n_saved, n_examples, latent_dim = paths.shape
    decoded = auto.decode(
        paths.reshape(-1, latent_dim).to(device)
    ).cpu().reshape(n_saved, n_examples, 1, 28, 28)

    fig, axes = plt.subplots(
        n_examples,
        n_saved,
        figsize=(1.4 * n_saved, 1.6 * n_examples),
        squeeze=False,
    )
    for row in range(n_examples):
        for col in range(n_saved):
            axes[row, col].imshow(decoded[col, row, 0], cmap='gray')
            axes[row, col].axis('off')
            if row == 0:
                axes[row, col].set_title(f"step {col}", fontsize=7)
    plt.tight_layout()
    plt.show()


In [ ]:
visualize_generator_walk(g_paths, auto_sae)

In [ ]:
visualize_generator_walk(g_paths, svae_net)

## Walk along each latent variable

In [ ]:
@torch.no_grad()
def latent_variable_walk(
    auto,
    x,
    spherical=False,
    dims=None,
    n_steps=100,
    step_size=1,
    span=3.0,
    device=device,
):
    """Walk from an encoded point along coordinate or tangent directions."""
    auto = auto.to(device).eval()

    while x.dim() < 4:
        x = x.unsqueeze(0)
    z0 = encode_for_walk(auto, x.to(device)).squeeze(0)

    if dims is None:
        dims = list(range(z0.numel()))
    else:
        dims = list(dims)

    offsets = torch.linspace(-span, span, n_steps, device=device)[::step_size]
    fig, axes = plt.subplots(
        len(dims),
        len(offsets),
        figsize=(1.5 * len(offsets), 1.8 * len(dims)),
        squeeze=False,
    )

    for row, dim in enumerate(dims):
        direction = torch.zeros_like(z0)
        direction[dim] = 1.0

        if not spherical:
            z_path = z0[None, :] + offsets[:, None] * direction[None, :]
        else:
            # Walk only on the S^2 factor containing this coordinate.
            z_blocks_raw = sphere_blocks(z0)
            radii = z_blocks_raw.norm(dim=-1, keepdim=True)
            z_blocks = z_blocks_raw / radii.clamp_min(1e-8)
            block_index = dim // SPHERE_BLOCK_DIM
            coordinate_index = dim % SPHERE_BLOCK_DIM
            base = z_blocks[block_index]

            block_direction = torch.zeros_like(base)
            block_direction[coordinate_index] = 1.0
            tangent = block_direction - torch.dot(block_direction, base) * base
            tangent_norm = tangent.norm()
            if tangent_norm < 1e-7:
                print(f"Skipping z{dim}: direction is radial at this point.")
                continue
            tangent = tangent / tangent_norm

            path_blocks = z_blocks_raw.unsqueeze(0).repeat(len(offsets), 1, 1)
            path_blocks[:, block_index] = radii[block_index] * (
                torch.cos(offsets)[:, None] * base[None, :]
                + torch.sin(offsets)[:, None] * tangent[None, :]
            )
            z_path = path_blocks.flatten(-2)

        reconstruction = auto.decode(z_path).cpu()
        for col, offset in enumerate(offsets):
            axes[row, col].imshow(reconstruction[col, 0], cmap='gray')
            axes[row, col].axis('off')
            axes[row, col].set_title(f"{offset.item():.2f}", fontsize=7)
            if col == 0:
                axes[row, col].set_ylabel(f"z{dim}", rotation=0, labelpad=20)

    plt.tight_layout()
    plt.show()


In [ ]:
x = test_images[2]
latent_variable_walk(
    svae_net,
    x,
    spherical=True,
    dims=range(latent_size),
    n_steps=200,
    step_size=10,
    span=np.pi, #3.0,
)


In [ ]:
x = test_images[5]
latent_variable_walk(
    vae_net,
    x,
    spherical=False,
    dims=range(latent_size),
    n_steps=200,
    step_size=10,
    span=3.0,
)


In [ ]:
@torch.no_grad()
def visualize_g_walk(
    auto,
    generator,
    x,
    n_steps=36,
    epsilon=0.1,
    device=device,
):
    """Integrate and visualize either Euclidean or spherical G."""
    paths = generator_walk(
        generator,
        auto,
        x,
        n_steps=n_steps,
        epsilon=epsilon,
        save_every=1,
        device=device,
    )
    visualize_generator_walk(paths, auto, device=device)

    print("initial norm:", paths[0, 0].norm().item())
    print("final norm:", paths[-1, 0].norm().item())
    print("distance first -> last:", torch.norm(paths[0, 0] - paths[-1, 0]).item())
    return paths


In [ ]:
ae_g_walk = visualize_g_walk(
    auto_ae,
    generator,
    test_images[10],
    n_steps=36,
    epsilon=0.1,
    device=device,
)


In [ ]:
x = test_images[7]
latent_variable_walk(
    svae_net,
    x,
    spherical=True,
    dims=range(svae_net.z_dim),
    n_steps=200,
    step_size=10,
    span=np.pi,
)


In [ ]:
# Use the same G-walk function for an S-VAE after training a spherical G
# on the S-VAE latent codes, for example:
svae_g_walk = visualize_g_walk(
    svae_net,
    generator,
    test_images[0],
    n_steps=36,
    epsilon=0.1,
    device=device,
)
